# **Section 7: Summary - Classification**

Everything runs on its own, the data is
built in the notebook, so you can change a number and re-run.


#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- **Chapter 17 of the textbook.**

**Where this is used.** This section feeds **Lab 7** (distance, the train/test
split, and the single nearest neighbour) and then **Project 2**, which goes
further: it votes among `k = 11` neighbours and asks you to write the vote
yourself.


In [ ]:
# Run this cell first -- the install takes about a minute in the browser
%pip install -q datascience ipywidgets

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

---

## **Contents**

1. [Regression or classification?](#1)
2. [The training set](#2)
3. [Distance between two points](#3)
4. [Row objects](#4)
5. [The nearest neighbour](#5)
6. [k nearest neighbours](#6)
7. [Train and test](#7)
8. [Accuracy](#8)
9. [Decision boundaries](#9)
10. [**>>Quick questions<<**](#10)
11. [**>>Self-check<<**](#11)
12. [Quick reference](#12)


---

<a id='1'></a>
## **1. Regression or classification?**

Both predict an outcome from attributes. The difference is what kind of thing
you are predicting.

| | Predicts | Example |
|---|---|---|
| **Regression** | a number | how many marks will this student get? |
| **Classification** | a category | will this student pass or fail? |

Section 6 did the first. This Section does the second, and the method is completely
different, no line, no formula, just: **find similar cases and see what
happened to them.**


---

<a id='2'></a>
## **2. The training set**

A classifier learns from examples whose class you already know. Each row is one
individual: some **attributes** you can measure, and a **class** you are trying
to predict.


In [ ]:
# 50 patients: two measurements, and whether they had the condition
np.random.seed(17)
n = 25
healthy = Table().with_columns(
    'Marker A', np.round(np.random.normal(3.5, 1.15, n), 2),
    'Marker B', np.round(np.random.normal(3.0, 1.15, n), 2),
    'Class',    np.repeat('no', n))
ill = Table().with_columns(
    'Marker A', np.round(np.random.normal(5.0, 1.15, n), 2),
    'Marker B', np.round(np.random.normal(4.5, 1.15, n), 2),
    'Class',    np.repeat('yes', n))
# with_rows stacks two tables. It is not on the reference sheet, and it is only
# used here to build the example data, so you will not need to write it.
patients = healthy.with_rows(ill.rows)
patients.show(4)

In [ ]:
patients.scatter('Marker A', 'Marker B', group='Class')
plots.title('The training set')
plots.show()

The two classes sit in different regions, but they **overlap**: there is no
line that separates them cleanly. That is what real data looks like, and it is
why the choice of k will matter.


---

<a id='3'></a>
## **3. Distance between two points**

"Similar" means "close", and close needs a definition. In two dimensions it is
the ordinary straight-line distance:

$$D = \sqrt{(x_0 - x_1)^2 + (y_0 - y_1)^2}$$

In more dimensions the same formula just gains more terms. It is called
**Euclidean distance**.


In [ ]:
def distance(point1, point2):
    """The Euclidean distance between two arrays of coordinates."""
    return np.sqrt(sum((point1 - point2) ** 2))

a = make_array(3, 4)
b = make_array(0, 0)
print('distance:', distance(a, b))

In [ ]:
# It works in any number of dimensions
p = make_array(1, 2, 3, 4)
q = make_array(2, 4, 6, 8)
print('4-D distance:', round(distance(p, q), 4))

> **Scale matters enormously.** If one attribute runs 0-10 and another 0-10,000,
> the second dominates the distance completely and the first is ignored. Convert
> both to standard units first, the same `standard_units` from Week 6, whenever
> the attributes are measured in different units.


---

<a id='4'></a>
## **4. Row objects**

`tbl.row(i)` gives you one row as an object. To compute distances you need its
attribute values as an array, and `np.array(list(row))` does that, but only
after dropping any non-numerical columns.


In [ ]:
attributes = patients.drop('Class')
first_row = attributes.row(0)
print('the row: ', first_row)
print('as array:', np.array(list(first_row)))

In [ ]:
def row_to_array(row):
    """Converts a row of numerical attributes into an array."""
    return np.array(list(row))

def distance_from(row, point):
    """Distance between one row of attributes and a point."""
    return distance(row_to_array(row), point)

new_patient = make_array(4.4, 4.0)
print('distance from row 0:', round(distance_from(attributes.row(0), new_patient), 3))

---

<a id='5'></a>
## **5. The nearest neighbour**

The simplest classifier possible: find the single closest training point and
copy its class.


In [ ]:
def all_distances(training, point):
    """Distance from every training row to a point."""
    attribute_table = training.drop('Class')
    return make_array(*[distance_from(attribute_table.row(i), point)
                        for i in np.arange(attribute_table.num_rows)])

with_dists = patients.with_columns('Distance', all_distances(patients, new_patient))
with_dists.sort('Distance').show(5)

In [ ]:
nearest = with_dists.sort('Distance').column('Class').item(0)
print('the new patient is nearest to a patient with Class =', nearest)

That works, and it is fragile. One unusual training point next to your new case
decides the answer entirely. Looking at several neighbours fixes that.


---

<a id='6'></a>
## **6. k nearest neighbours**

Take the **k** closest points and let them vote. The majority class wins.

The whole algorithm is four steps:

1. Compute the distance to every training point
2. Sort by distance
3. Take the first k
4. Return the most common class among them


In [ ]:
def closest(training, point, k):
    """The k nearest training rows to a point."""
    with_dists = training.with_columns('Distance', all_distances(training, point))
    return with_dists.sort('Distance').take(np.arange(k))

def majority_class(topk):
    """The most common class in a table of neighbours."""
    return topk.group('Class').sort('count', descending=True).column('Class').item(0)

def classify(training, point, k):
    """Classify a point by majority vote of its k nearest neighbours."""
    return majority_class(closest(training, point, k))

print('k=1 :', classify(patients, new_patient, 1))
print('k=5 :', classify(patients, new_patient, 5))
print('k=11:', classify(patients, new_patient, 11))

### **One thing to watch: these names differ in Lab 7 and Project 2**

`distance` is the same everywhere, and it is the one you write most often. The
others are named differently in each place.

| here and in the lectures | Lab 7 | Project 2 |
|---|---|---|
| `distance(point1, point2)` | `distance(features1, features2)` | `distance(features1, features2)` |
| `all_distances(training, point)` | `distances_from(test_row, training_table)` | `fast_distances(test_row, train_table)` |
| `closest(training, point, k)` | not used, Lab 7 stops at one neighbour | `closest_ten` |
| `majority_class(topk)` | shown once, not written | `most_common(label, table)` |
| `classify(training, point, k)` | not used | `classify(test_row, train_features, train_labels, k)` |

Most of those are **given** to you, so the name changing does not matter. Two do
matter, because **you write them in Project 2**: `most_common` and `classify`.
Note that Project 2's `classify` takes the test row first, then the training
features and labels separately, then `k`. The version here takes the training
table first.

### **Choosing k**

**Odd**, so a two-class vote cannot tie.

**Small k** follows the training data closely, including its noise. **Large k**
smooths, but with k large enough you simply predict the commonest class every
time, ignoring the point entirely.

There is no formula. You try several and measure.


---

<a id='7'></a>
## **7. Train and test**

Here is the thing students get wrong.

**A classifier tested on its own training data will look better than it is.**
With k=1 it scores 100%, every point is its own nearest neighbour.

So split the data. **Train on one part, test on the other**, and never let the
classifier see the test rows while it learns.


In [ ]:
shuffled = patients.sample(with_replacement=False)
train = shuffled.take(np.arange(35))
test  = shuffled.take(np.arange(35, patients.num_rows))
print('train:', train.num_rows, ' test:', test.num_rows)

Shuffle before splitting. The table was built with all the healthy patients
first, take the first 35 rows without shuffling and your training set is almost
entirely one class.


---

<a id='8'></a>
## **8. Accuracy**

The proportion of test points the classifier gets right.


In [ ]:
def classify_table(training, test_table, k):
    """Classify every row of a test table."""
    attrs = test_table.drop('Class')
    return make_array(*[classify(training, row_to_array(attrs.row(i)), k)
                        for i in np.arange(test_table.num_rows)])

def accuracy(training, test_table, k):
    predicted = classify_table(training, test_table, k)
    return np.average(predicted == test_table.column('Class'))

for k in [1, 3, 5, 7, 9]:
    print(f'k = {k}: accuracy {accuracy(train, test, k):.0%}')

Look at what happens between k=1 and k=3. **k=1 does noticeably worse**. It is
following individual training points, including the ones that sit in the wrong
region. Averaging over three neighbours smooths that out.

That is the tradeoff the previous section described, visible in a number.

> **Accuracy alone can mislead.** If 95% of your data is one class, a classifier
> that always guesses that class scores 95% and is useless. Always check what
> accuracy a constant guess would achieve before being impressed by a number.


In [ ]:
# The accuracy of guessing the commonest class, every time
commonest = test.group('Class').sort('count', descending=True).column('Class').item(0)
print(f"always guessing '{commonest}': "
      f"{np.average(test.column('Class') == commonest):.0%}")

---

<a id='9'></a>
## **9. Decision boundaries**

Classify every point on a grid and you can see the regions the classifier has
carved out. The line between them is the **decision boundary**.


In [ ]:
grid_a, grid_b, grid_class = [], [], []
for a_val in np.arange(1.0, 8.1, 0.35):
    for b_val in np.arange(0.5, 7.6, 0.35):
        grid_a.append(a_val)
        grid_b.append(b_val)
        grid_class.append(classify(patients, make_array(a_val, b_val), 5))

grid = Table().with_columns('Marker A', grid_a, 'Marker B', grid_b,
                            'Class', grid_class)
grid.scatter('Marker A', 'Marker B', group='Class', alpha=0.35, s=12)
plots.title('The regions a k=5 classifier assigns')
plots.show()

The boundary is **not a straight line**. k-NN makes no assumption about shape, 
it follows wherever the training points lead, which is its main advantage over
methods that impose a form in advance.

Try k=1 in the loop above and re-run. The boundary becomes ragged, with small
islands around individual training points. That is overfitting, made visible.


---

<a id='10'></a>
## **10. Quick questions**

Five of them. Set `my_answer` to a letter and run the cell.
A wrong answer gets a nudge so you can try again; a right one gets the reason.

These come before the written questions below on purpose: they are quicker,
and they check the things people most often get wrong.

In [ ]:
# Run this once. mcq.py must be in the same folder as this notebook.
from mcq import check_answer, show_answer

**M1.** Why is a classifier not evaluated on the rows it learned from?

**a)** it takes too long  
**b)** the training rows are not representative  
**c)** it has already seen those answers, so the score means nothing  
**d)** it is, that is the usual practice  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w7_m1', my_answer)

**M2.** One attribute runs 0 to 10 and another 0 to 10,000. What should you do before measuring distance?

**a)** convert both to standard units  
**b)** drop the larger one  
**c)** nothing, distance handles it  
**d)** divide the larger by 1000  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w7_m2', my_answer)

**M3.** Why is an odd `k` usually preferred with two classes?

**a)** it runs faster  
**b)** an even `k` can tie  
**c)** odd numbers give smoother boundaries  
**d)** there is no reason; it is convention  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w7_m3', my_answer)

**M4.** Your classifier gets 76% right. Is that good?

**a)** yes, anything above 70% is good  
**b)** no, you need 90% or better  
**c)** yes, if the test set is large enough  
**d)** you cannot tell without the baseline  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w7_m4', my_answer)

**M5.** Why must the class column be dropped before computing distances?

**a)** it makes the calculation slower  
**b)** distance only works on two columns  
**c)** distance is measured on the attributes; including the answer is circular  
**d)** it does not have to be  

In [ ]:
my_answer = '?'          # a, b, c or d
check_answer('w7_m5', my_answer)

---

<a id='11'></a>
## **11. Self-check**

Answer these without scrolling back. Reveal each answer only after you have committed to one.

**Q1.** You want to predict how long a shot flew, in metres. Is that classification?

<details>
<summary><strong>Answer</strong></summary>

No, that is <strong>regression</strong>: the answer is a number on a continuous scale. Classification predicts which <strong>category</strong> something belongs to, from a fixed set of labels.

</details>

**Q2.** Why must the class column be dropped before computing distances?

<details>
<summary><strong>Answer</strong></summary>

Distance is measured on the <strong>attributes</strong>. Leaving the class in either crashes on a non-numerical value, or, worse, quietly lets the answer influence the measurement of similarity, which is circular.

</details>

**Q3.** Why is a classifier not evaluated on the same rows it learned from?

<details>
<summary><strong>Answer</strong></summary>

Because it has already seen those answers, so its accuracy on them flatters it. A 1-nearest-neighbour classifier scores 100% on its own training set by finding each point's identical self. Accuracy only means something on rows the classifier has never seen.

</details>

**Q4.** Why is an odd k usually preferred with two classes?

<details>
<summary><strong>Answer</strong></summary>

An even k can <strong>tie</strong>, with the same number of neighbours voting each way, and the classifier then needs an arbitrary rule to break it. An odd k cannot tie between two classes.

</details>

**Q5.** The summary calls the vote `majority_class(topk)`. Project 2 calls it `most_common(label, table)`. Do they do the same thing?

<details>
<summary><strong>Answer</strong></summary>

Yes: both group the neighbours by class, sort by count and return the commonest label. The names and argument orders differ across the lectures, Lab 7 and Project 2. Write whichever the question in front of you asks for, and check the argument order rather than assuming.

</details>

**Q6.** Your classifier gets 76% right. Why is that not enough information to say it is any good?

<details>
<summary><strong>Answer</strong></summary>

You need the <strong>baseline</strong>: how often you would be right always guessing the commonest class. If 74% of the data is one class, then 76% is almost nothing. A classifier is only worth having if it beats always guessing.

</details>

---

<a id='12'></a>
## **12. Quick reference**

### **The whole classifier**

```python
def distance(point1, point2):
    return np.sqrt(sum((point1 - point2) ** 2))

def closest(training, point, k):
    dists = ...                                  # distance to every row
    return training.with_columns('Distance', dists).sort('Distance').take(np.arange(k))

def majority_class(topk):
    return topk.group('Class').sort('count', descending=True).column('Class').item(0)

def classify(training, point, k):
    return majority_class(closest(training, point, k))
```

### **Calls**

| Call | Gives |
|---|---|
| `tbl.row(i)` | one row as an object |
| `np.array(list(row))` | that row's values as an array |
| `tbl.drop('Class')` | attributes only, needed before computing distances |
| `tbl.sample(with_replacement=False)` | a shuffle, for splitting |
| `tbl.take(np.arange(n))` | the first n rows |

### **Things that catch people out**

| | |
|---|---|
| Attributes on different scales | the big one dominates; convert to standard units |
| Testing on training data | k=1 always scores 100% and means nothing |
| Splitting without shuffling | you may train on one class only |
| Even k | a two-class vote can tie |
| Accuracy on its own | compare it to always guessing the commonest class |
| Forgetting `.drop('Class')` | the class column ends up in the distance |

---

### **Textbook**

- [Chapter 17, Classification](https://inferentialthinking.com/chapters/17/classification/)
- [Chapter 17.1, Nearest neighbours](https://inferentialthinking.com/chapters/17/1/nearest-neighbors/)
- [Chapter 17.3, Rows of tables](https://inferentialthinking.com/chapters/17/3/rows-of-tables/)
- [Chapter 17.4, Implementing the classifier](https://inferentialthinking.com/chapters/17/4/implementing-the-classifier/)
- [Chapter 17.5, The accuracy of the classifier](https://inferentialthinking.com/chapters/17/5/accuracy-of-the-classifier/)
